# Parcial 1

Nohelanys Camaño, Kevin Linares, Juan Morales y Ruben Feng

1. Cargamos el CSV


In [ ]:
# Carga del dataset
import pandas as pd

df = pd.read_csv('Balaji Fast Food Sales (1).csv')
df.head()

2. DEA Inicial

In [ ]:
# Análisis exploratorio inicial


print('Filas:', df.shape[0], '| Columnas:', df.shape[1])
df.info()

#Nulos y duplicados
print('\nDuplicados:', df.duplicated().sum())
print('Duplicados sin order_id:', df.drop(columns='order_id').duplicated().sum())
print('\nNulos:\n', df.isnull().sum())

# Valores únicos de las columnas categóricas
for col in ['item_name', 'item_type', 'transaction_type', 'received_by', 'time_of_sale']:
    print('\n', df[col].value_counts(dropna=False))

# Muestra de fechas para revisar formatos
print('\n', df['date'].sample(10, random_state=1))

# Resumen estadístico
df.describe()

No existen duplicados relevantes, solo existen nulos con respecto al tipo de pago, solo existen dos: en efectivo y en línea, podríamos asumir que el nulo que falta podría ser de tarjeta pero no se sabe y no se pude saber.

3. Diccionarios para el dataset (idioma)

In [ ]:
# Diccionarios de traducción al español
columnas = {'order_id': 'id_orden', 'date': 'fecha', 'item_name': 'producto',
            'item_type': 'tipo_producto', 'item_price': 'precio', 'quantity': 'cantidad',
            'transaction_amount': 'monto', 'transaction_type': 'medio_pago',
            'received_by': 'atendido_por', 'time_of_sale': 'turno'}

productos = {'Cold coffee': 'Café frío', 'Sugarcane juice': 'Jugo de caña', 'Sandwich': 'Sándwich',
             'Panipuri': 'Panipuri', 'Aalopuri': 'Aalopuri', 'Vadapav': 'Vadapav', 'Frankie': 'Frankie'}

tipos_producto = {'Fastfood': 'Comida rápida', 'Beverages': 'Bebidas'}
medios_pago = {'Cash': 'Efectivo', 'Online': 'En línea'}
atendido_por = {'Mr.': 'Sr.', 'Mrs.': 'Sra.'}

# Turnos en orden lógico a lo largo del día
turnos = {'Morning': 'Mañana', 'Afternoon': 'Tarde', 'Evening': 'Anochecer',
          'Night': 'Noche', 'Midnight': 'Medianoche'}



Muestra del diccionario

In [ ]:
# Ejemplo: tabla completa traducida
df_es = df.copy()
df_es['item_name'] = df_es['item_name'].map(productos)
df_es['item_type'] = df_es['item_type'].map(tipos_producto)
df_es["transaction_type"] = df_es["transaction_type"].fillna("Desconocido")
df_es['transaction_type'] = df_es['transaction_type'].map(medios_pago)
df_es['received_by'] = df_es['received_by'].map(atendido_por)
df_es['time_of_sale'] = pd.Categorical(df_es['time_of_sale'].map(turnos),
                                       categories=list(turnos.values()), ordered=True)
df_es = df_es.rename(columns=columnas)

# Mostrar todas las filas
pd.set_option('display.max_rows', 10)
df_es

# Reportes

Reporte por medio de pago: cantidad de ventas y monto total en efectivo y en línea. Tambien mostrar porcentajes (cantidad de ventas y montos). 

In [ ]:

# Cantidad de ventas por medio de pago
print("Cantidad de ventas en Lineas:", df_es[df_es['medio_pago'] == 'En línea']['cantidad'].count())
print("Cantidad de ventas en efectivo:", df_es[df_es['medio_pago'] == 'Efectivo']['cantidad'].count())

# Monto total por medio de pago
print("Monto total en linea :", df_es[df_es['medio_pago'] == 'En línea']['monto'].sum())
print("Monto total en efectivo :", df_es[df_es['medio_pago'] == 'Efectivo']['monto'].sum())

Reporte de productos por turno (por ejemplo, cuántos cafés fríos se vendieron en la mañana, tarde, noche y medianoche). Escoja al menos 3 productos para presentar.

In [ ]:
producto_busqueda = ["Aalopuri", "Vadapav", "Frankie"]

for producto in producto_busqueda:
    print("Cantidad de ventas de", producto, ":\n", df_es[df_es['producto'] == producto].groupby("turno")["cantidad"].sum())
    print("\t\t")

Reporte de tipo de producto por medio de pago.  

In [ ]:
# Reporte de tipo de producto por medio de pago

reporte_tipo_pago = (
  df_es.assign(medio_pago=df_es['medio_pago'].fillna('Desconocido'))
  .groupby(['tipo_producto', 'medio_pago'])
  .agg(
    unidades=('cantidad', 'sum'),
    ventas=('id_orden', 'count'),
    monto=('monto', 'sum')
  )
  .unstack(fill_value=0)
)

# Agregar totales por tipo de producto
reporte_tipo_pago['Total'] = reporte_tipo_pago.sum(axis=1)

# Agregar total general
reporte_tipo_pago.loc['Total'] = reporte_tipo_pago.sum(axis=0)

reporte_tipo_pago

Reporte de producto más vendido( por unidades y por ingreso) de cada mes

In [ ]:
# Convertir fechas y crear el mes de cada venta
fechas = pd.to_datetime(df_es['fecha'], errors='coerce', format='mixed')
df_mensual = df_es.assign(
  mes=fechas.dt.to_period('M').astype(str)
)

# Resumen mensual por producto
resumen_mensual = (
  df_mensual
  .groupby(['mes', 'producto'], as_index=False)
  .agg(
    unidades=('cantidad', 'sum'),
    ingreso=('monto', 'sum')
  )
)

# Producto más vendido por unidades e ingreso en cada mes
mas_unidades = (
  resumen_mensual.loc[
    resumen_mensual.groupby('mes')['unidades'].idxmax()
  ]
  .rename(columns={
    'producto': 'producto_mas_unidades',
    'unidades': 'max_unidades'
  })
  [['mes', 'producto_mas_unidades', 'max_unidades']]
)

mas_ingreso = (
  resumen_mensual.loc[
    resumen_mensual.groupby('mes')['ingreso'].idxmax()
  ]
  .rename(columns={
    'producto': 'producto_mas_ingreso',
    'ingreso': 'max_ingreso'
  })
  [['mes', 'producto_mas_ingreso', 'max_ingreso']]
)

reporte_mensual = (
  mas_unidades
  .merge(mas_ingreso, on='mes')
  .sort_values('mes')
)
pd.set_option('display.max_rows', None)
reporte_mensual

Reporte por medio de pago: Cantidad de ventas y monto total en efectivo y en linea. Porcentaje de cantidades de ventas y monto

productos que hay que tener en inventario por quincea, efevtivo en caja por turno

o	Reporte por medio de pago: cantidad de ventas y monto total en efectivo y en línea. Tambien mostrar porcentajes (cantidad de ventas y montos). 
o	Reporte de productos por turno (por ejemplo, cuántos cafés fríos se vendieron en la mañana, tarde, noche y medianoche). Escoja al menos 3 productos para presentar. 
o	Reporte de tipo de producto por medio de pago.  
o	Reporte de producto más vendido( por unidades y por ingreso) de cada mes
